In [ ]:
from collections import Counter
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

1. 学習データ読み込み

In [ ]:
file_path = "../data/raw/train-00000-of-00001.parquet"
df_raw = pd.read_parquet(file_path)

2.前処理

In [ ]:
df_train = df_raw.copy()

In [ ]:
# データ型の確認
df_train.dtypes

In [ ]:
col_names_one_hot_encoding = [
    "job", "marital", "education", "contact", "month", "poutcome","poutcome"
]

col_names_binary = [
    "default", "housing", "loan", "y"
]

In [ ]:
# 前処理関数
# 学習データとテストデータで同一にする
# baselineでは正則化はしない

def preprocessing(df):
    
    # onehotencoding
    # カテゴリー型に変更→onehotencdingの順番で実施
    for col in col_names_one_hot_encoding:
        df[col] = df[col].astype("category")

    df = pd.get_dummies(df, columns=col_names_one_hot_encoding)

    # binary(yes or no)のカラムをboolに変換
    for col in col_names_binary:
        df[col] = df[col].str.lower().map({"yes": True, "no": False})
    
    # print(df.head(5))

    return df

In [ ]:
df_train = preprocessing(df=df_train)

In [ ]:
df_train.head(5)

3.学習

3.1 データセット作成

In [ ]:
df_train.columns

In [ ]:
y_col = "y"

X_train = df_train.drop(y_col, axis=1)
y_train = df_train[y_col]

# gbm系と違って.fitの時に検証ができないため、この時点で分割しない
# gridsearchとかもベースラインではやらない
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


3.2 学習の実施

In [ ]:
# ロジスティック回帰
model_log_reg = LogisticRegression()
model_log_reg.fit(X_train, y_train)

In [ ]:
# SVM
model_svm = SVC(kernel="rbf") 
model_svm.fit(X_train, y_train)

4.予測&評価

4.1 テストデータの準備

In [ ]:
file_path = "../data/raw/test-00000-of-00001.parquet"
df_test = pd.read_parquet(file_path)

In [ ]:
df_test = preprocessing(df=df_test)

In [ ]:
X_test = df_test.drop(y_col, axis=1)
y_test = df_test[y_col]

4.2 予測精度確認

In [ ]:
def predict_and_calculate_metrics(model, X_test, y_test):

    y_pred_prob = model.predict(X_test)
    y_pred = [1 if i >= 0.5 else 0 for i in y_pred_prob]
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

    # yが不均衡データなので、予測値の中身を確認
    counts = Counter(y_pred)
    print(counts)
    print("予測値のうち、yesの割合:", counts[1]/(counts[0]+counts[1]))

    # 実際のyのTrue, Falseを見る
    df_test["y"].value_counts(normalize=True)

    print(f"precision_score: {precision_score(y_test, y_pred, pos_label=True)}")
    print(f"recall_score: {recall_score(y_test, y_pred, pos_label=True)}")
    print(f"f1_score: {f1_score(y_test, y_pred, pos_label=True)}")


In [ ]:
# ロジスティック回帰 
# model name → model_log_reg

predict_and_calculate_metrics(model_log_reg, X_test, y_test)

# accuracyは0.89だけど、recallが0.22
# 予測対象が不均衡データの影響

In [ ]:
# SVM
# model name → model_svm

predict_and_calculate_metrics(model_svm, X_test, y_test)

# accuracyは0.885だけど、recallが0.007
# 予測対象が不均衡データの影響